[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-Bayoumy/MECH559/blob/colab-version/L06/L06_stabilization_and_scaling.ipynb)

# Ensuring Descent: Hessian Modification and Scaling
### MECH 559 - Systems Optimization - L06
### Course Instructor: Dr. Ahmed Bayoumy
Author of the notebook: Dr. Ahmed Bayoumy

Newton-type methods only work when $H_k$ is positive definite. This notebook demonstrates what goes wrong when it isn't (converging to a saddle instead of a minimizer), how modifying the Hessian fixes it, and how poor variable scaling can wreck a well-conditioned-looking problem - and how to fix that too.

## Learning objectives

By the end, students should be able to:

1. Explain why $M_k$ (the matrix in $x_{k+1}=x_k-\alpha_k M_k \nabla f_k$) must be positive (semi-)definite for guaranteed descent.
2. Modify an indefinite Hessian into a positive-definite one and verify the resulting direction is a genuine descent direction.
3. Reproduce the lecture's "Newton converges to a saddle without modification, to a minimizer with it" example.
4. Compute a Hessian's condition number, apply diagonal scaling, and see the dramatic effect on gradient-descent convergence speed.

> **Classroom rhythm:** pause at each **Predict** prompt, collect answers, then run the next cell.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3D projection)

try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

COLORS = {"boundary": "#00798C", "optimum": "#D1495B", "accent": "#EDAE49", "bad": "0.5"}

print(f"NumPy {np.__version__}; optional widgets available: {WIDGETS_AVAILABLE}")

---
## 1. A modified Hessian gives a genuine descent direction

For the lecture's example, $f(x_1,x_2)=2x_1^2-4x_1x_2+1.5x_2^2+x_2$, the (constant) Hessian is $H=\begin{bmatrix}4&-4\\-4&3\end{bmatrix}$, which is **indefinite** (its determinant is $12-16=-4<0$). We regularize it by shifting its eigenvalues up to a minimum positive floor - a simple, robust stand-in for modified Cholesky - and check that the resulting direction actually decreases $f$.

In [ ]:
def f_quad2(x):
    x1, x2 = x
    return 2*x1**2 - 4*x1*x2 + 1.5*x2**2 + x2

def grad_quad2(x):
    x1, x2 = x
    return np.array([4*x1 - 4*x2, -4*x1 + 3*x2 + 1])

H = np.array([[4.0, -4.0], [-4.0, 3.0]])
print("H eigenvalues:", np.linalg.eigvalsh(H), "-> indefinite (one negative eigenvalue)")

def modify_hessian(H, min_eig=0.5):
    w, V = np.linalg.eigh(H)
    w_mod = np.maximum(w, min_eig)
    return V @ np.diag(w_mod) @ V.T

H_hat = modify_hessian(H)
print("modified Hessian:\n", np.round(H_hat, 3))
print("modified eigenvalues:", np.linalg.eigvalsh(H_hat), "-> positive definite")

x0 = np.array([0.0, 0.0])
g0 = grad_quad2(x0)
d_raw = np.linalg.solve(H, -g0)
d_mod = np.linalg.solve(H_hat, -g0)
print(f"\nraw Newton direction:      {np.round(d_raw,4)}, grad^T d = {g0@d_raw: .3f} (>=0 means NOT a descent direction!)")
print(f"modified Newton direction: {np.round(d_mod,4)}, grad^T d = {g0@d_mod: .3f} (descent direction)")

---
## 2. Reproducing the lecture's saddle-vs-minimum example

$$
f(x_1,x_2) = \tfrac13 x_1^3 + x_1x_2 + \tfrac12x_2^2 + 2x_2 - \tfrac23
$$

This function has two stationary points: a genuine local minimizer at $(2,-4)$ (Hessian positive definite there) and a saddle point at $(-1,-1)$ (Hessian indefinite there). Starting Newton's method near the saddle's basin, **without** Hessian modification it converges to the saddle; **with** modification it escapes to the real minimizer.

> **Predict:** before running the cell, guess which of the two critical points plain Newton's method will find starting from $x_0=(0,0)$.

In [ ]:
def f_cubic(x):
    x1, x2 = x
    return (1/3)*x1**3 + x1*x2 + 0.5*x2**2 + 2*x2 - 2/3

def grad_cubic(x):
    x1, x2 = x
    return np.array([x1**2 + x2, x1 + x2 + 2])

def hess_cubic(x):
    x1, x2 = x
    return np.array([[2*x1, 1.0], [1.0, 1.0]])

for xc, name in [((2.0, -4.0), "candidate A"), ((-1.0, -1.0), "candidate B")]:
    xc = np.array(xc)
    Hc = hess_cubic(xc)
    kind = "positive definite (minimizer)" if np.all(np.linalg.eigvalsh(Hc) > 0) else "indefinite (saddle point)"
    print(f"{name} {xc}: grad={grad_cubic(xc)}, Hessian eigenvalues={np.round(np.linalg.eigvalsh(Hc),3)} -> {kind}")

def newton_raw(x0, n_iter=15, tol=1e-8):
    x = np.array(x0, dtype=float)
    traj = [x.copy()]
    for _ in range(n_iter):
        g = grad_cubic(x)
        if np.linalg.norm(g) < tol:
            break
        H = hess_cubic(x)
        d = np.linalg.solve(H, -g)
        x = x + d
        traj.append(x.copy())
    return np.array(traj)

def newton_modified(x0, n_iter=15, tol=1e-8, min_eig=0.5):
    x = np.array(x0, dtype=float)
    traj = [x.copy()]
    for _ in range(n_iter):
        g = grad_cubic(x)
        if np.linalg.norm(g) < tol:
            break
        H_hat = modify_hessian(hess_cubic(x), min_eig=min_eig)
        d = np.linalg.solve(H_hat, -g)
        alpha = 1.0
        while f_cubic(x + alpha*d) > f_cubic(x) + 1e-4*alpha*(g @ d) and alpha > 1e-10:
            alpha *= 0.5
        x = x + alpha * d
        traj.append(x.copy())
    return np.array(traj)

x0 = [0.0, 0.0]
traj_raw = newton_raw(x0)
traj_mod = newton_modified(x0)
print(f"\nwithout modification: converges to {np.round(traj_raw[-1],4)} in {len(traj_raw)-1} iterations")
print(f"with modification:    converges to {np.round(traj_mod[-1],4)} in {len(traj_mod)-1} iterations")

x1g, x2g = np.meshgrid(np.linspace(-3, 4, 200), np.linspace(-6, 2, 200))
Z = (1/3)*x1g**3 + x1g*x2g + 0.5*x2g**2 + 2*x2g - 2/3
fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(x1g, x2g, Z, levels=40, cmap="viridis", alpha=0.6)
ax.plot(traj_raw[:, 0], traj_raw[:, 1], "-o", color=COLORS["bad"], label="without Hessian modification -> saddle")
ax.plot(traj_mod[:, 0], traj_mod[:, 1], "-o", color=COLORS["boundary"], label="with Hessian modification -> minimizer")
ax.scatter([2], [-4], marker="*", s=200, color=COLORS["optimum"], zorder=5, label="true minimizer (2,-4)")
ax.scatter([-1], [-1], marker="x", s=120, color="black", zorder=5, label="saddle point (-1,-1)")
ax.legend(fontsize=8)
ax.set(xlabel="$x_1$", ylabel="$x_2$", title="Effect of Hessian modification on Newton's method")
plt.tight_layout()
plt.show()

---
## 3. Condition number and scaling

$$
f(x_1,x_2) = 1000x_1^2 + 40x_1x_2 + x_2^2, \qquad H = \begin{bmatrix}2000&40\\40&2\end{bmatrix}.
$$

Diagonal scaling with $D_{ii}=1/|H_{ii}|$, $H_{\text s}=D^{1/2}HD^{1/2}$, brings the condition number down enormously.

In [ ]:
H_bad = np.array([[2000.0, 40.0], [40.0, 2.0]])
eigs_bad = np.linalg.eigvalsh(H_bad)
kappa_bad = eigs_bad[-1] / eigs_bad[0]
print(f"H eigenvalues: {eigs_bad}, condition number kappa = {kappa_bad:.1f}")

D = np.diag(1 / np.abs(np.diag(H_bad)))
D_sqrt = np.diag(np.sqrt(np.diag(D)))
H_scaled = D_sqrt @ H_bad @ D_sqrt
eigs_scaled = np.linalg.eigvalsh(H_scaled)
kappa_scaled = eigs_scaled[-1] / eigs_scaled[0]
print(f"D = diag({np.diag(D)}); H_scaled =\n{np.round(H_scaled,3)}")
print(f"scaled eigenvalues: {np.round(eigs_scaled,4)}, condition number kappa = {kappa_scaled:.2f}")

def gd_exact_quadratic(A, x0, n_iter=500, tol=1e-10):
    x = np.array(x0, dtype=float)
    for k in range(n_iter):
        g = A @ x
        if g @ g < tol:
            return k, x
        alpha = (g @ g) / (g @ A @ g)
        x = x - alpha * g
    return n_iter, x

x0_slow = np.array([0.0407109, -1.37801453])   # a deliberately "unlucky" starting direction
n_unscaled, xf_unscaled = gd_exact_quadratic(H_bad, x0_slow)
D_inv_sqrt = np.linalg.inv(D_sqrt)
y0 = D_inv_sqrt @ x0_slow
n_scaled, yf_scaled = gd_exact_quadratic(H_scaled, y0)

print(f"\nsame starting difficulty, unscaled problem:  {n_unscaled} iterations to converge")
print(f"same starting difficulty, scaled problem:    {n_scaled} iterations to converge")

---
## 4. Interactive: how scaling reshapes the contours

Move the off-diagonal coupling term and watch how elongated (and hard to optimize) the unscaled contours become, versus the always-round scaled ones.

In [ ]:
def scaling_demo(h11=2000.0, h22=2.0, h12=40.0):
    H_demo = np.array([[h11, h12], [h12, h22]])
    eigs = np.linalg.eigvalsh(H_demo)
    if eigs[0] <= 0:
        print("Not positive definite for these values - pick h11, h22 > 0 with |h12| small enough.")
        return
    kappa = eigs[-1] / eigs[0]
    D_demo = np.diag(1 / np.abs(np.diag(H_demo)))
    Ds = np.diag(np.sqrt(np.diag(D_demo)))
    Hs = Ds @ H_demo @ Ds
    kappa_s = np.linalg.cond(Hs)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    for ax, A, title, kap in [(axes[0], H_demo, "unscaled", kappa), (axes[1], Hs, "scaled", kappa_s)]:
        lim = 1.5
        x1g, x2g = np.meshgrid(np.linspace(-lim, lim, 200), np.linspace(-lim, lim, 200))
        Z = 0.5*(A[0,0]*x1g**2 + 2*A[0,1]*x1g*x2g + A[1,1]*x2g**2)
        ax.contour(x1g, x2g, Z, levels=20, cmap="viridis")
        ax.set_title(f"{title}: kappa={kap:.1f}")
        ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()

if WIDGETS_AVAILABLE:
    display(widgets.interactive(scaling_demo,
                                 h11=widgets.FloatLogSlider(value=2000, base=10, min=1, max=4, step=0.1),
                                 h22=widgets.FloatLogSlider(value=2, base=10, min=-1, max=2, step=0.1),
                                 h12=widgets.FloatSlider(value=40, min=0, max=60, step=2)))
else:
    print("ipywidgets is not installed; running the default case.")
    scaling_demo()

---
## Takeaways

1. $M_k$ in $x_{k+1}=x_k-\alpha_kM_k\nabla f_k$ must be positive (semi-)definite for guaranteed descent; an indefinite Hessian can turn "Newton's method" into an *ascent* step.
2. A simple eigenvalue-floor regularization (a stand-in for modified Cholesky) restores a valid descent direction whenever the raw Hessian isn't positive definite.
3. On a function with both a genuine minimizer and a saddle point, unmodified Newton's method can converge to the *saddle* - modification reliably redirects it to the true minimizer.
4. Condition number ($\kappa=\lambda_{\max}/\lambda_{\min}$) measures how elongated the objective's contours are; simple diagonal scaling can cut $\kappa$ by orders of magnitude and correspondingly speed up gradient-based methods by orders of magnitude (here, 203 iterations down to 10, from the very same starting difficulty).

## Optional exercises

1. Try `min_eig` values of 0.01, 0.5, and 5 in `modify_hessian` for the saddle example. Does the algorithm still escape the saddle in every case? How does `min_eig` affect the step size taken?
2. Compute the condition number of $H=\begin{bmatrix}2000&40\\40&2\end{bmatrix}$ using `np.linalg.cond` directly and confirm it matches the eigenvalue-ratio calculation.
3. For the scaling demo, find values of $h_{12}$ (with $h_{11}=2000,h_{22}=2$ fixed) at which $H$ stops being positive definite. Relate this to the leading-principal-minors test from the earlier FONC/SOSC lecture.